# Post Pandemic Regime Shifts in Labor Market: Data Fetch and Merge

## Purpose

This notebook fetches data from FRED and other external sources to build a single date-aligned dataset for structural break analysis and forecasting. The purpose is to ensure that labor market tightness, wage and inflation measures, and supporting macro features are all present in the merged dataset. Diagnostics are also run to ensure the data is pulled accurately, that vintage years are handled properly, and that older data is managed well.

## Research Context

The scope of this project is limited to the United States. The goal of this project is to determine whether the job-openings-to-unemployment ratio (labor market tightness) experienced a structural break in its relationship with future wage growth and inflation after the 2020 pandemic shock, and whether a new predictive signal emerged in the post-break regime.

To answer this question, both historical and current economic data are needed. For this project, Federal Reserve Economic Data, Atlanta Fed Wage Growth Tracker files, and Philadelphia Fed vintage workbooks are used.

## Inputs and Outputs
The notebook consumes:
* raw monthly data from FRED,
* the Atlanta Fed Wage Growth Tracker file,
* and Philadelphia Fed vintage workbooks.

The output is the saved merged dataset. The other outputs, such as the merge log and missingness tables, are for diagnostic purposes that help determine whether later estimated breaks reflect economic instability or change in data availability.

## Imports and Configuration

In [1]:
%matplotlib inline

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 50)

project_root = Path.cwd()
if not (project_root / "regime_shift").exists():
    for root in [Path.cwd(), *Path.cwd().parents]:
        if (root / "regime_shift").exists():
            project_root = root
            break

sys.path.insert(0, str(project_root))

import importlib

from regime_shift.config import ATL_MAP, DICT_PATH, MERGED_PATH, MERGE_PATH, RAW_ROOT
from regime_shift.data import FileLoader, FredLoader
from regime_shift.preprocessing import (
    check_panel,
    data_dict,
    load_file,
    merge_data,
    missing_data,
    process_vintage,
    set_date,
)

fred_root = RAW_ROOT / "fred"
file_root = RAW_ROOT / "external"

fred_root.mkdir(parents=True, exist_ok=True)
file_root.mkdir(parents=True, exist_ok=True)

2026-04-13 13:05:22,460 - regime_shift - INFO - Initialized regime_shift package (v0.0.1)


## Data Loading

Before beginning data loading, go to the Federal Reserve Economic Data (FRED) website, sign up, and obtain an API key. Once you have it, paste it where indicated (“insert key here”) and run the code. Otherwise, the FRED data will not load. If you don't want to get an API key, use the fred_panel in data/processed or the merged file and continue to the EDA.

### Fetch FRED Data

In [3]:
api_key = os.getenv("FRED_API_KEY", " insert key here").strip()

if not api_key:
    raise ValueError("FRED API key is missing.")

In [4]:
series_map = {
    "job_openings_level": "JTSJOL",
    "hires_rate": "JTSHIR",
    "hires_level": "JTSHIL",
    "quits_rate": "JTSQUR",
    "quits_level": "JTSQUL",
    "total_separations_rate": "JTSTSR",
    "total_separations_level": "JTSTSL",
    "layoffs_discharges_level": "JTSLDL",
    "layoffs_discharges_rate": "JTSLDR",
    "unemployment_rate": "UNRATE",
    "unemployment_level": "UNEMPLOY",
    "u6_rate": "U6RATE",
    "prime_age_lfpr": "LNS11300060",
    "lfpr": "CIVPART",
    "epop_ratio": "EMRATIO",
    "prime_age_urate": "LNS13025703",
    "avg_weeks_unemployed": "UEMPMEAN",
    "payrolls_nonfarm": "PAYEMS",
    "payrolls_private": "USPRIV",
    "payrolls_manufacturing": "MANEMP",
    "payrolls_services": "SRVPRD",
    "payrolls_construction": "USCONS",
    "ahe_private": "CES0500000003",
    "awe_private": "CES0500000011",
    "awh_private": "AWHAETP",
    "awe_manufacturing": "CES3000000008",
    "ahe_manufacturing": "CES3000000003",
    "eci_total": "ECIALLCIV",
    "eci_wages": "ECIWAG",
    "cpi_all": "CPIAUCSL",
    "cpi_core": "CPILFESL",
    "cpi_less_shelter": "CUSR0000SA0L2",
    "cpi_services_less_rent": "CUUR0000SASL2RS",
    "cpi_services_less_energy": "CUSR0000SASLE",
    "cpi_shelter": "CUSR0000SEHA",
    "cpi_medical": "CPIMEDSL",
    "cpi_food": "CPIUFDSL",
    "pce_price": "PCEPI",
    "pce_core": "PCEPILFE",
    "pce_trimmed_12m": "PCETRIM12M159SFRBDAL",
    "pce_trimmed_1m": "PCETRIM1M158SFRBDAL",
    "pce_services": "DSERRG3M086SBEA",
    "saving_rate": "PMSAVE",
    "real_dpi": "DSPIC96",
    "real_pi": "RPI",
    "real_pi_less_transfers": "W875RX1",
    "retail_advance": "RSAFS",
    "real_retail": "RRSFS",
    "ip_total": "INDPRO",
    "ip_manufacturing": "IPMAN",
    "capacity_util": "CUMFNS",
    "housing_starts": "HOUST",
    "building_permits": "PERMIT",
    "new_home_sales": "HSN1F",
    "consumer_sentiment": "UMCSENT",
    "durable_orders": "DGORDER",
    "capex_orders": "NEWORDER",
    "ci_loans": "BUSLOANS",
    "bank_credit": "TOTBKCR",
    "m2": "M2SL",
    "treasury_1m": "DGS1MO",
    "treasury_3m": "DGS3MO",
    "treasury_2y": "DGS2",
    "treasury_10y": "GS10",
    "fed_funds": "FEDFUNDS",
    "aaa_yield": "AAA",
    "baa_yield": "BAA",
    "hy_oas": "BAMLH0A0HYM2",
    "bbb_oas": "BAMLC0A4CBBB",
    "breakeven_5y": "T5YIE",
    "breakeven_10y": "T10YIE",
    "forward_5y5y": "T5YIFR",
    "michigan_1y": "MICH",
    "michigan_5y": "MICH5Y",
    "kcfed_lmci_activity": "FRBKCLMCILA",
    "kcfed_lmci_momentum": "FRBKCLMCIM",
    "recession": "USREC",
    "output_per_hour": "OPHNFB",
    "unit_labor_costs": "ULCNFB",
}

In [5]:
start_date = "2000-01-01"
end_date = pd.Timestamp.today().normalize().strftime("%Y-%m-%d")

In [5]:
loader = FredLoader(api_key)

fred_panel, fred_meta, fred_fail = loader.pull_many(
    series_map=series_map,
    start_date=start_date,
    end_date=end_date,
)

fred_panel.to_csv(fred_root / "01_fred_panel.csv")
fred_meta.to_csv(fred_root / "01_fred_meta.csv", index=False)
fred_fail.to_csv(fred_root / "01_fred_fail.csv", index=False)

print(f"FRED Rows: {fred_panel.shape[0]}")
print(f"FRED Cols: {fred_panel.shape[1]}")
print(f"Data Range: {fred_panel.index.min().date()} to {fred_panel.index.max().date()}")

2026-04-12 15:55:33,815 - regime_shift.data - INFO - [1/79] Pulling job_openings_level
2026-04-12 15:55:35,175 - regime_shift.data - INFO - [2/79] Pulling hires_rate
2026-04-12 15:55:35,879 - regime_shift.data - INFO - [3/79] Pulling hires_level
2026-04-12 15:55:36,537 - regime_shift.data - INFO - [4/79] Pulling quits_rate
2026-04-12 15:55:37,307 - regime_shift.data - INFO - [5/79] Pulling quits_level
2026-04-12 15:55:38,046 - regime_shift.data - INFO - [6/79] Pulling total_separations_rate
2026-04-12 15:55:38,719 - regime_shift.data - INFO - [7/79] Pulling total_separations_level
2026-04-12 15:55:39,402 - regime_shift.data - INFO - [8/79] Pulling layoffs_discharges_level
2026-04-12 15:55:40,099 - regime_shift.data - INFO - [9/79] Pulling layoffs_discharges_rate
2026-04-12 15:55:40,939 - regime_shift.data - INFO - [10/79] Pulling unemployment_rate
2026-04-12 15:55:41,915 - regime_shift.data - INFO - [11/79] Pulling unemployment_level
2026-04-12 15:55:42,636 - regime_shift.data - INFO -

FRED Rows: 316
FRED Cols: 78
Data Range: 2000-01-01 to 2026-04-01


### Fetching Data from External Sources

In [6]:
url_map = {
    "atlanta_wage_tracker": "https://www.atlantafed.org/-/media/Project/Atlanta/FRBA/Documents/datafiles/chcs/wage-growth-tracker/wage-growth-data.xlsx",
    "philly_ruc_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/rucQvMd.xlsx",
    "philly_cpi_vintages": "https://www.philadelphiafed.org/-/media/FRBP/Assets/Surveys-And-Data/real-time-data/data-files/xlsx/cpiQvMd.xlsx",
}

file_map, file_log = FileLoader.fetch_many(url_map=url_map, out_root=file_root)
file_log.to_csv(file_root / "01_file_log.csv", index=False)

print(file_log)

                   file                                                url                                               path status  size_bytes
0  atlanta_wage_tracker  https://www.atlantafed.org/-/media/Project/Atl...  /Users/Sumaitat/Documents/Coding/labor-market-...     ok      335748
1   philly_ruc_vintages  https://www.philadelphiafed.org/-/media/FRBP/A...  /Users/Sumaitat/Documents/Coding/labor-market-...     ok      567097
2   philly_cpi_vintages  https://www.philadelphiafed.org/-/media/FRBP/A...  /Users/Sumaitat/Documents/Coding/labor-market-...     ok      590396


In [7]:
def set_date(
    df: pd.DataFrame,
    date_col: str = "date",
    month_start: bool = True,
) -> pd.DataFrame:
    work = df.copy()

    if date_col not in work.columns:
        raise ValueError("Date column is missing.")

    work[date_col] = pd.to_datetime(work[date_col], errors="coerce")
    work = work.dropna(subset=[date_col]).copy()

    period = work[date_col].dt.to_period("M")

    if month_start:
        work[date_col] = period.dt.to_timestamp(how="start")
    else:
        work[date_col] = period.dt.to_timestamp(how="end")

    work = work.sort_values(date_col).drop_duplicates(subset=[date_col], keep="last")
    work = work.reset_index(drop=True)

    return work

## Data Integration

In [8]:
fred_data = load_file(
    fred_root / "01_fred_panel.csv",
    file_type="csv",
    index_col="date",
    parse_dates=True,
)

if fred_data is None:
    raise ValueError("FRED data is missing.")

fred_merge = fred_data.reset_index()
fred_merge["date"] = pd.to_datetime(fred_merge["date"], errors="coerce")
fred_merge = set_date(fred_merge, date_col="date")

fred_merge.to_csv(RAW_ROOT / "processed_fred.csv", index=False)

print(f"FRED Rows: {fred_merge.shape[0]}")
print(f"FRED Columns: {fred_merge.shape[1]}")

FRED Rows: 316
FRED Columns: 79


In [9]:
atl_merge = None
atl_path = file_root / "atlanta_wage_tracker.xlsx"

if atl_path.exists():
    atl_data = pd.read_excel(
        atl_path,
        sheet_name="data_overall",
        engine="openpyxl",
        skiprows=2,
    )

    date_col = None
    best_pct = 0.0

    for col in atl_data.columns:
        test = pd.to_datetime(atl_data[col], errors="coerce")
        test_pct = test.notna().mean()
        if test_pct > best_pct and test_pct >= 0.80:
            date_col = col
            best_pct = test_pct

    if date_col is None:
        raise ValueError("Date column is missing in Atlanta data.")

    atl_data["date"] = pd.to_datetime(atl_data[date_col], errors="coerce")

    if date_col != "date":
        atl_data = atl_data.drop(columns=[date_col])

    atl_data = atl_data.dropna(subset=["date"]).copy()
    atl_data = set_date(atl_data, date_col="date")

    for col in atl_data.columns:
        if col == "date":
            continue
        atl_data[col] = pd.to_numeric(atl_data[col].replace(".", np.nan), errors="coerce")

    rename_map = {}
    for col in atl_data.columns:
        if col == "date":
            continue
        rename_map[col] = ATL_MAP.get(col, f"atl_{col}")

    atl_merge = atl_data.rename(columns=rename_map)
    atl_merge.to_csv(RAW_ROOT / "processed_atlanta.csv", index=False)

    print(f"Atlanta Rows: {atl_merge.shape[0]}")
    print(f"Atlanta Cols: {atl_merge.shape[1]}")

else:
    print("Atlanta file couldn't be found, try again.")

Atlanta Rows: 350
Atlanta Cols: 17


In [10]:
ruc_merge = None
ruc_path = file_root / "philly_ruc_vintages.xlsx"

if ruc_path.exists():
    ruc_raw = pd.read_excel(ruc_path, sheet_name="ruc", engine="openpyxl")
    ruc_data, ruc_years = process_vintage(
        df=ruc_raw,
        col_pattern=r"RUC(\d{2})Q\d",
        prefix="phl_ruc",
        keep_last=5,
    )

    if ruc_data is not None:
        ruc_merge = set_date(ruc_data, date_col="date")
        ruc_merge.to_csv(RAW_ROOT / "processed_philly_ruc.csv", index=False)

        print(f"RUC Rows: {ruc_merge.shape[0]}")
        print(f"RUC Cols: {ruc_merge.shape[1]}")
        print(f"RUC Years: {ruc_years}")
else:
    print("RUC file couldn't be found, try again.")

RUC Rows: 949
RUC Cols: 21
RUC Years: [2015, 2016, 2017, 2018, 2019]


In [11]:
cpi_merge = None
cpi_path = file_root / "philly_cpi_vintages.xlsx"

if cpi_path.exists():
    cpi_raw = pd.read_excel(cpi_path, sheet_name="cpi", engine="openpyxl")
    cpi_data, cpi_years = process_vintage(
        df=cpi_raw,
        col_pattern=r"CPI(\d{2})Q\d",
        prefix="phl_cpi",
        keep_last=5,
    )

    if cpi_data is not None:
        cpi_merge = set_date(cpi_data, date_col="date")
        cpi_merge.to_csv(RAW_ROOT / "processed_philly_cpi.csv", index=False)

        print(f"CPI Rows: {cpi_merge.shape[0]}")
        print(f"CPI Cols: {cpi_merge.shape[1]}")
        print(f"CPI Years: {cpi_years}")
else:
    print("CPI file couldn't be found, try again.")

CPI Rows: 949
CPI Cols: 21
CPI Years: [2015, 2016, 2017, 2018, 2019]


## Data Concatenation

In [ ]:
data_map = {
    "atlanta": atl_merge,
    "philly_ruc": ruc_merge,
    "philly_cpi": cpi_merge,
}

In [13]:
merged_data, merge_log = merge_data(
    base_df=fred_merge,
    data_map=data_map,
    date_col="date",
    how="left",
)

merged_data["date"] = pd.to_datetime(merged_data["date"])
merged_data = merged_data.sort_values("date").reset_index(drop=True)

print(f"Final Rows: {merged_data.shape[0]}")
print(f"Final Columnss: {merged_data.shape[1]}")
print(f"Data Range: {merged_data['date'].min().date()} to {merged_data['date'].max().date()}")

Final Rows: 316
Final Columnss: 135
Data Range: 2000-01-01 to 2026-04-01


In [14]:
panel_check = check_panel(merged_data, date_col="date")
panel_check

,metric,value
0,Rows,316
1,Cols,135
2,Start Date,2000-01-01
3,End Date,2026-04-01
4,Full Months,316
5,Missing Months,0


## Data Validation

In [15]:
complete_cases = merged_data.dropna(how="any").shape[0]
complete_pct = (complete_cases / len(merged_data)) * 100
print(f"\nComplete Cases: {complete_cases} ({complete_pct:.2f}%)")


Complete Cases: 106 (33.54%)


In [16]:
miss_data = missing_data(merged_data)
miss_data.head(20)

,col,missing_count,missing_pct,non_null_count,data_type
0,phl_cpi_CPI15Q1,136,43.04,180,float64
1,phl_ruc_RUC15Q1,135,42.72,181,float64
2,phl_cpi_CPI15Q2,133,42.09,183,float64
3,phl_ruc_RUC15Q2,132,41.77,184,float64
4,phl_cpi_CPI15Q3,130,41.14,186,float64
5,phl_ruc_RUC15Q3,129,40.82,187,float64
6,phl_cpi_CPI15Q4,127,40.19,189,float64
7,phl_ruc_RUC15Q4,126,39.87,190,float64
8,phl_cpi_CPI16Q1,124,39.24,192,float64
9,phl_ruc_RUC16Q1,123,38.92,193,float64


In [17]:
complete_rows = merged_data.dropna(how="any").shape[0]
complete_pct = round(100 * complete_rows / len(merged_data), 2)

print(f"Complete Rows: {complete_rows}")
print(f"Complete Share: {complete_pct}%")

Complete Rows: 106
Complete Share: 33.54%


In [18]:
period_map = {
    "Pre-2010": merged_data[merged_data["date"] < "2010-01-01"],
    "2010-2019": merged_data[
        (merged_data["date"] >= "2010-01-01") & (merged_data["date"] < "2020-01-01")
    ],
    "2020+": merged_data[merged_data["date"] >= "2020-01-01"],
}

,period,rows,complete_rows,complete_pct
0,Pre-2010,120,46,38.33
1,2010-2019,120,60,50.00
2,2020+,76,0,0.00


In [ ]:
period_rows = []
for name, data in period_map.items():
    rows = len(data)
    full_rows = data.dropna(how="any").shape[0]
    full_pct = round(100 * full_rows / rows, 2) if rows > 0 else np.nan

    period_rows.append(
        {
            "period": name,
            "rows": rows,
            "complete_rows": full_rows,
            "complete_pct": full_pct,
        }
    )

In [ ]:
period_df = pd.DataFrame(period_rows)
period_df

## Final Dataset

In [19]:
data_dictionary = data_dict(merged_data)

merge_log.to_csv(MERGE_PATH, index=False)
miss_data.to_csv(RAW_ROOT / "06_missing_data.csv", index=False)
panel_check.to_csv(RAW_ROOT / "06_panel_check.csv", index=False)
period_df.to_csv(RAW_ROOT / "06_period_check.csv", index=False)

merged_data.to_csv(MERGED_PATH, index=False)
data_dictionary.to_csv(DICT_PATH, index=False)

In [23]:
summary_data = pd.DataFrame(
    [
        {"metric": "fred_rows", "value": fred_merge.shape[0]},
        {"metric": "fred_cols", "value": fred_merge.shape[1]},
        {"metric": "atl_rows", "value": atl_merge.shape[0] if atl_merge is not None else 0},
        {"metric": "atl_cols", "value": atl_merge.shape[1] if atl_merge is not None else 0},
        {"metric": "ruc_rows", "value": ruc_merge.shape[0] if ruc_merge is not None else 0},
        {"metric": "ruc_cols", "value": ruc_merge.shape[1] if ruc_merge is not None else 0},
        {"metric": "cpi_rows", "value": cpi_merge.shape[0] if cpi_merge is not None else 0},
        {"metric": "cpi_cols", "value": cpi_merge.shape[1] if cpi_merge is not None else 0},
        {"metric": "final_rows", "value": merged_data.shape[0]},
        {"metric": "final_cols", "value": merged_data.shape[1]},
        {"metric": "start_date", "value": str(merged_data["date"].min().date())},
        {"metric": "end_date", "value": str(merged_data["date"].max().date())},
        {"metric": "complete_rows", "value": complete_rows},
        {"metric": "complete_pct", "value": complete_pct},
    ]
)

summary_data.to_csv(MERGE_PATH, index=False)
summary_data

,metric,value
0,fred_rows,316
1,fred_cols,79
2,atl_rows,350
3,atl_cols,17
4,ruc_rows,949
5,ruc_cols,21
6,cpi_rows,949
7,cpi_cols,21
8,final_rows,316
9,final_cols,135


In [24]:
print(f"Merged File: {MERGED_PATH}")
print(f"Data Dictionary: {DICT_PATH}")
print(f"Merge File: {MERGE_PATH}")

Merged File: /Users/Sumaitat/Documents/Coding/labor-market-regime-shift/data/processed/07_merged_data.csv
Data Dict: /Users/Sumaitat/Documents/Coding/labor-market-regime-shift/data/processed/07_data_dictionary.csv
Merge File: /Users/Sumaitat/Documents/Coding/labor-market-regime-shift/data/processed/07_merge_summary.csv


In [2]:
df = pd.read_csv(MERGED_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Data Shape: {df.shape}")
print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")

Data Shape: (316, 135)
Date Range: 2000-01-01 to 2026-04-01


## Summary

In this notebook, raw data is pulled from several external sources. Many of the websites were updated, placing files deeper within the sites and requiring more careful handling. It was easier to manually download the data and place it into an external file than to pull it directly from the sites. The vintage years had to be modified to fit the date structure. In this notebook, the data is then pulled and merged into a consolidated dataset. The merged file will be used for EDA to answer the question.